# 08 — GNN Autoencoder per Anomaly Detection di Emerging Jets

**Task 2 del progetto** — architettura GNN.

## Idea chiave
Ogni jet è un **grafo**: i nodi sono le tracce valide, i lati le connettono
in spazio (Δη, Δφ) con k-NN. Un **Graph Autoencoder** (GAE) impara la struttura
dei jet QCD ordinari. I jet EJ, con tracce *displaced*, avranno un errore di
ricostruzione più alto → **anomaly score**.

## Architettura
```
Encoder : EdgeConv [24→64] → EdgeConv [64→128] → Global Mean Pool → z ∈ R^{32}
Decoder : Concat(z_broadcast, h_node) → MLP [160→128→64→24]
```

## Input
- `outputs/cleaning_out/cleaning_config.yaml` + `norm_stats.npz`  (da preprocess_fin)
- `pp_output_test_background.h5` (QCD) e `pp_output_test_signal.h5` (EJ)

## Output → `outputs/GNN/`
- `gnn_ae_weights.pt` — pesi del modello
- `gnn_loss.png` — curve di loss
- `gnn_scores_dist.png` — distribuzioni anomaly score QCD vs EJ
- `gnn_roc.png` — curva ROC
- `gnn_test_scores.npz` — scores salvati per confronto con AE/VAE


In [2]:
import importlib, subprocess, sys

pkgs = {"torch_geometric": "torch_geometric", "pyyaml": "yaml"}
for pkg, import_name in pkgs.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Dipendenze installate.")

Dipendenze installate.


In [3]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "pyyaml", "-q"],
    capture_output=True, text=True, timeout=60
)
print(result.stdout, result.stderr)

In [4]:
import os, gc, time, warnings, json
from pathlib import Path

import numpy as np
import h5py
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import torch_geometric
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader as PyGDataLoader
    from torch_geometric.nn import EdgeConv, global_mean_pool, knn_graph
    HAS_PYG = True
    print(f"torch_geometric {torch_geometric.__version__}  ✓")
except ImportError:
    HAS_PYG = False
    print("ERRORE: torch_geometric non trovato. Installa con: pip install torch_geometric")
    raise ImportError("torch_geometric richiesto per questo notebook.")

print(f"PyTorch {torch.__version__}")
warnings.filterwarnings("ignore")


torch_geometric 2.7.0  ✓
PyTorch 2.11.0+cpu


In [5]:
# ── Percorsi ──────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"C:\\Users\\delco\\Desktop\\Universita\\Ad_M\\Dataset-Programs")
PATH_BKG  = BASE_DIR / "pp_output_test_background.h5"
PATH_SIG  = BASE_DIR / "pp_output_test_signal.h5"
CLEAN_DIR = BASE_DIR / "outputs" / "cleaning_out"
OUT_DIR   = BASE_DIR / "outputs" / "GNN"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PATH_BKG, PATH_SIG, CLEAN_DIR / "cleaning_config.yaml", CLEAN_DIR / "norm_stats.npz"]:
    status = "✓" if Path(p).exists() else "✗ NON TROVATO"
    print(f"  {Path(p).name}: {status}")

# ── Iperparametri ──────────────────────────────────────────────────────────────
N_SAMPLES_PER_FILE = 50_000   # jet per file H5
MAX_TRACKS         = 128      # padding (P99 ≈ 125 tracce valide)
KNN_K              = 8        # vicini per il grafo k-NN
LATENT_DIM         = 32       # dimensione spazio latente
HIDDEN_DIMS        = [64, 128]# canali dei layer EdgeConv
BATCH_SIZE         = 256      # jet per batch
EPOCHS             = 60       # epoche di training
LR                 = 1e-3     # learning rate Adam
WEIGHT_DECAY       = 1e-5
SEED               = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nessuna'}")


  pp_output_test_background.h5: ✓
  pp_output_test_signal.h5: ✓
  cleaning_config.yaml: ✓
  norm_stats.npz: ✓

Device: cpu
GPU: nessuna


In [6]:
# Carica la configurazione prodotta da preprocess_fin.ipynb
with open(CLEAN_DIR / "cleaning_config.yaml") as f:
    cfg = yaml.safe_load(f)

norm = np.load(CLEAN_DIR / "norm_stats.npz", allow_pickle=True)
MEAN           = norm["mean"].astype(np.float32)
STD            = norm["std"].astype(np.float32)
TRACK_FEATURES = cfg["track_input_features"]
TRACK_CLIP     = {k: tuple(v) for k, v in cfg["track_clip"].items()}
SENTINEL       = float(cfg["sentinel_value"])
N_FEAT         = len(TRACK_FEATURES)

# Indici delle feature spaziali per il k-NN graph
DETA_IDX = TRACK_FEATURES.index("deta")
DPHI_IDX = TRACK_FEATURES.index("dphi")

print(f"Feature di traccia: {N_FEAT}")
print(f"deta index={DETA_IDX}  |  dphi index={DPHI_IDX}")
print(f"\nPrime 5 feature: {TRACK_FEATURES[:5]}")
print(f"MEAN range: [{MEAN.min():.4f}, {MEAN.max():.4f}]")
print(f"STD  range: [{STD.min():.4f},  {STD.max():.4f}]")


Feature di traccia: 24
deta index=3  |  dphi index=2

Prime 5 feature: ['d0_log1p', 'z0SinTheta', 'dphi', 'deta', 'qOverP']
MEAN range: [-0.0403, 98.3704]
STD  range: [0.0000,  118.8319]


## 1. Caricamento e preprocessing dei dati

Carichiamo i jet dai file H5, applichiamo il **cleaning identico** a `preprocess_fin.ipynb`
(sentinella → 0, clipping, masking) e la **normalizzazione z-score** calcolata SOLO sui QCD.


In [7]:
def load_and_preprocess(path: Path, n_samples: int, label: int) -> dict:
    """
    Carica n_samples jet dal file H5, applica cleaning + normalizzazione.

    Returns dict con chiavi:
      'tracks_norm' : float32 (N, MAX_TRACKS, N_FEAT)  — tracce normalizzate
      'masks'       : bool    (N, MAX_TRACKS)           — True = traccia valida
      'labels'      : int64   (N,)                      — 0=QCD, 1=EJ
      'event_nums'  : int64   (N,)                      — eventNumber originale
    """
    with h5py.File(path, "r") as f:
        n_tot = f["jets"].shape[0]
        n = min(n_samples, n_tot)
        print(f"  Carico {n:,}/{n_tot:,} jet da {path.name} (label={label})...", end=" ", flush=True)
        t0 = time.time()
        jets_raw   = f["jets"][:n]
        tracks_raw = f["tracks"][:n]
    print(f"{time.time()-t0:.1f}s")

    # Maschera delle tracce valide
    valid_mask = tracks_raw["valid"][:, :MAX_TRACKS].astype(bool)       # (N, MAX_TRACKS)

    # Costruzione tensore tracce: stack delle feature selezionate
    X = np.stack(
        [tracks_raw[feat][:, :MAX_TRACKS].astype(np.float32) for feat in TRACK_FEATURES],
        axis=-1
    )  # (N, MAX_TRACKS, N_FEAT)

    # Cleaning: sentinella → 0, clipping, nan/inf → 0
    for i, feat in enumerate(TRACK_FEATURES):
        col = X[:, :, i]
        col[col == SENTINEL] = 0.0
        if feat in TRACK_CLIP:
            lo, hi = TRACK_CLIP[feat]
            col = np.clip(col, lo, hi)
        X[:, :, i] = np.nan_to_num(col, nan=0.0, posinf=0.0, neginf=0.0)

    # Azzera le posizioni di padding
    X[~valid_mask] = 0.0

    # Normalizzazione z-score (MEAN/STD calcolati solo su QCD in preprocess_fin)
    X = (X - MEAN) / STD
    X[~valid_mask] = 0.0          # ri-azzera padding dopo normalizzazione

    # Rimuovi jet completamente privi di tracce valide
    n_valid = valid_mask.sum(axis=1)
    keep = n_valid > 0
    print(f"  Rimossi {(~keep).sum()} jet vuoti. Rimasti: {keep.sum():,}")

    event_nums = jets_raw["eventNumber"][keep].astype(np.int64)
    labels     = np.full(keep.sum(), label, dtype=np.int64)

    return {
        "tracks_norm": X[keep].astype(np.float32),
        "masks":       valid_mask[keep],
        "labels":      labels,
        "event_nums":  event_nums,
    }


print("Caricamento QCD (background)...")
data_bkg = load_and_preprocess(PATH_BKG, N_SAMPLES_PER_FILE, label=0)
print("Caricamento EJ (signal)...")
data_sig = load_and_preprocess(PATH_SIG, N_SAMPLES_PER_FILE, label=1)

print(f"\nQCD: {data_bkg['tracks_norm'].shape[0]:,} jet")
print(f"EJ : {data_sig['tracks_norm'].shape[0]:,} jet")


Caricamento QCD (background)...
  Carico 50,000/1,000,000 jet da pp_output_test_background.h5 (label=0)... 3.6s
  Rimossi 0 jet vuoti. Rimasti: 50,000
Caricamento EJ (signal)...
  Carico 50,000/61,966 jet da pp_output_test_signal.h5 (label=1)... 4.3s
  Rimossi 0 jet vuoti. Rimasti: 50,000

QCD: 50,000 jet
EJ : 50,000 jet


In [8]:
def split_by_event(data: dict, split: str, seed: int = SEED) -> dict:
    """
    Split deterministico basato su indice (80/10/10).
    Non usa eventNumber % 10 perché in questo dataset tutti gli event number
    terminano con la stessa cifra (artefatto della numerazione MC).
    """
    n = len(data["labels"])
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    if split == "train":
        sel = idx[:n_train]
    elif split == "val":
        sel = idx[n_train:n_train + n_val]
    elif split == "test":
        sel = idx[n_train + n_val:]
    else:
        raise ValueError(f"split sconosciuto: {split}")
    return {k: v[sel] for k, v in data.items()}


# QCD: train + val (unsupervised → niente signal in training)
qcd_train = split_by_event(data_bkg, "train")
qcd_val   = split_by_event(data_bkg, "val")

# Test: QCD test + EJ test (per valutazione anomaly detection)
qcd_test  = split_by_event(data_bkg, "test")
ej_test   = split_by_event(data_sig, "test")

print(f"Split QCD   →  train: {len(qcd_train['labels']):,}  |  val: {len(qcd_val['labels']):,}  |  test: {len(qcd_test['labels']):,}")
print(f"Split EJ    →  test:  {len(ej_test['labels']):,}")

Split QCD   →  train: 40,000  |  val: 5,000  |  test: 5,000
Split EJ    →  test:  5,000


## 2. Classe Dataset: grafi jet

Per ogni jet costruiamo un oggetto `torch_geometric.data.Data`:
- **Nodi** `x` : solo le tracce valide (niente padding) — shape `(n_valid, N_FEAT)`
- **Lati** `edge_index` : k-NN in spazio (Δη, Δφ) — shape `(2, n_edges)`
- `y` : label del jet (0=QCD, 1=EJ)


In [9]:
from torch.utils.data import Dataset

def _knn_graph(pos: torch.Tensor, k: int) -> torch.Tensor:
    """Build k-NN edge_index (source_to_target flow) without torch-cluster."""
    dist = torch.cdist(pos, pos)                   # (n, n)
    dist.fill_diagonal_(float('inf'))
    _, nbrs = dist.topk(k, largest=False, dim=1)   # (n, k)
    nodes = torch.arange(pos.size(0)).unsqueeze(1).expand_as(nbrs).reshape(-1)
    return torch.stack([nbrs.reshape(-1), nodes], dim=0)  # (2, n*k)


class JetGraphDataset(Dataset):
    """
    Ogni __getitem__ restituisce un torch_geometric.data.Data con:
      x          : (n_valid, N_FEAT) tracce normalizzate
      edge_index : (2, E) grafo k-NN in (deta, dphi)
      y          : label scalare (0=QCD, 1=EJ)
    """

    def __init__(self, tracks_norm: np.ndarray, masks: np.ndarray,
                 labels: np.ndarray, knn_k: int = KNN_K):
        self.tracks = tracks_norm   # (N, MAX_TRACKS, N_FEAT)
        self.masks  = masks         # (N, MAX_TRACKS)
        self.labels = labels        # (N,)
        self.k      = knn_k

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x_pad = self.tracks[idx]           # (MAX_TRACKS, N_FEAT)
        valid  = self.masks[idx]           # (MAX_TRACKS,) bool
        valid_idx = np.where(valid)[0]

        # Estrai solo tracce valide (rimuovi padding)
        x = x_pad[valid_idx].astype(np.float32)  # (n_valid, N_FEAT)
        n = len(x)

        if n == 0:
            x = np.zeros((1, N_FEAT), dtype=np.float32)
            n = 1

        x_t = torch.from_numpy(x)           # (n_valid, N_FEAT)

        # k-NN graph in spazio (deta, dphi)
        pos = x_t[:, [DETA_IDX, DPHI_IDX]]  # (n_valid, 2)
        k_eff = min(self.k, n - 1) if n > 1 else 0

        if k_eff > 0:
            edge_index = _knn_graph(pos, k=k_eff)  # (2, n_valid*k_eff)
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)

        return Data(
            x          = x_t,
            edge_index = edge_index,
            y          = torch.tensor(self.labels[idx], dtype=torch.long),
        )


# Crea i dataset
ds_train = JetGraphDataset(qcd_train["tracks_norm"], qcd_train["masks"], qcd_train["labels"])
ds_val   = JetGraphDataset(qcd_val["tracks_norm"],   qcd_val["masks"],   qcd_val["labels"])
ds_qtest = JetGraphDataset(qcd_test["tracks_norm"],  qcd_test["masks"],  qcd_test["labels"])
ds_etest = JetGraphDataset(ej_test["tracks_norm"],   ej_test["masks"],   ej_test["labels"])

print(f"Dataset → train: {len(ds_train):,}  |  val: {len(ds_val):,}")
print(f"          test QCD: {len(ds_qtest):,}  |  test EJ: {len(ds_etest):,}")

# Verifica un singolo grafo
sample = ds_train[0]
print(f"\nEsempio grafo:  x={tuple(sample.x.shape)}  edge_index={tuple(sample.edge_index.shape)}  y={sample.y.item()}")

Dataset → train: 40,000  |  val: 5,000
          test QCD: 5,000  |  test EJ: 5,000

Esempio grafo:  x=(128, 24)  edge_index=(2, 1024)  y=0


In [10]:
# torch_geometric DataLoader gestisce automaticamente il batching di grafi eterogenei
loader_train = PyGDataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val   = PyGDataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_qtest = PyGDataLoader(ds_qtest, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_etest = PyGDataLoader(ds_etest, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Verifica un batch
batch_ex = next(iter(loader_train))
print(f"Batch di esempio:")
print(f"  x:          {tuple(batch_ex.x.shape)}   (nodi_totali x N_FEAT)")
print(f"  edge_index: {tuple(batch_ex.edge_index.shape)}")
print(f"  batch:      {tuple(batch_ex.batch.shape)}  (indice jet per ogni nodo)")
print(f"  num_graphs: {batch_ex.num_graphs}")
del batch_ex; gc.collect()


Batch di esempio:
  x:          (16327, 24)   (nodi_totali x N_FEAT)
  edge_index: (2, 130616)
  batch:      (16327,)  (indice jet per ogni nodo)
  num_graphs: 256


40

## 3. GNN Autoencoder

### Encoder
Due blocchi **EdgeConv** con residual connection:
$$h^{(l+1)}_i = \max_{j \in \mathcal{N}(i)} \text{MLP}\bigl([h^{(l)}_i \| h^{(l)}_j - h^{(l)}_i]\bigr)$$

Poi **Global Mean Pool** sulle tracce valide → vettore latente `z` ∈ ℝ^{32}.

### Decoder
`z` viene broadcasting su ogni nodo e concatenato con le feature del nodo encoder;
un MLP ricostruisce le feature originali.

### Loss
MSE sulle feature di tutte le tracce nel batch (tutte valide, niente padding).


In [11]:
class EdgeConvBlock(nn.Module):
    """
    Un blocco EdgeConv con BN + ReLU.
    Input: concat(h_i, h_j - h_i)  →  shape (2*in_ch,)
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(2 * in_ch, out_ch),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Linear(out_ch, out_ch),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
        )
        self.conv = EdgeConv(mlp, aggr="mean")

    def forward(self, x, edge_index):
        return self.conv(x, edge_index)


class GNNAutoencoder(nn.Module):
    """
    Graph Autoencoder per anomaly detection su jet.

    Encoder
    -------
    EdgeConv[N_FEAT → 64] → EdgeConv[64 → 128] → Global Mean Pool → Linear[128→latent]

    Decoder
    -------
    Concat(z_broadcast, h_node) → MLP → x_rec (N_FEAT feature)
    """

    def __init__(self, n_feat: int, hidden: list, latent_dim: int):
        super().__init__()
        self.n_feat = n_feat

        # Encoder: stack di EdgeConvBlock
        self.enc_blocks = nn.ModuleList()
        in_ch = n_feat
        for out_ch in hidden:
            self.enc_blocks.append(EdgeConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.enc_last_ch = in_ch                   # = hidden[-1]

        # Proiezione → latente
        self.pool_proj = nn.Sequential(
            nn.Linear(self.enc_last_ch, latent_dim),
            nn.ReLU(),
        )

        # Decoder: MLP su (latent + last_node_features) → n_feat
        dec_in = latent_dim + self.enc_last_ch
        self.decoder = nn.Sequential(
            nn.Linear(dec_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_feat),
        )

    def encode(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = x
        for block in self.enc_blocks:
            h = block(h, edge_index)
        z_pool = global_mean_pool(h, batch)       # (B, enc_last_ch)
        z      = self.pool_proj(z_pool)           # (B, latent_dim)
        return h, z, batch

    def decode(self, h, z, batch):
        z_node = z[batch]                         # (N_nodes, latent_dim)
        h_dec  = torch.cat([z_node, h], dim=1)   # (N_nodes, latent_dim+enc_last_ch)
        return self.decoder(h_dec)                # (N_nodes, n_feat)

    def forward(self, data):
        h, z, batch = self.encode(data)
        x_rec = self.decode(h, z, batch)
        return x_rec, z, batch


# Istanza del modello
model = GNNAutoencoder(
    n_feat     = N_FEAT,
    hidden     = HIDDEN_DIMS,
    latent_dim = LATENT_DIM,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nParametri trainabili: {n_params:,}")


GNNAutoencoder(
  (enc_blocks): ModuleList(
    (0): EdgeConvBlock(
      (conv): EdgeConv(nn=Sequential(
        (0): Linear(in_features=48, out_features=64, bias=True)
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Linear(in_features=64, out_features=64, bias=True)
        (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()
      ))
    )
    (1): EdgeConvBlock(
      (conv): EdgeConv(nn=Sequential(
        (0): Linear(in_features=128, out_features=128, bias=True)
        (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Linear(in_features=128, out_features=128, bias=True)
        (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()
      ))
    )
  )
  (pool_proj): Sequential(
    (0): Linear(in_features=128, out_features=32, bias=Tr

## 4. Training

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total_jets = 0.0, 0
    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for batch in loader:
            batch = batch.to(DEVICE)
            x_rec, z, bvec = model(batch)

            # MSE su tutte le tracce del batch (già solo valide, niente padding)
            loss = F.mse_loss(x_rec, batch.x)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * batch.num_graphs
            total_jets += batch.num_graphs

    return total_loss / total_jets


train_losses, val_losses = [], []
best_val_loss = float("inf")
best_state    = None
t_start = time.time()

print(f"{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>12}  {'LR':>10}  {'Time':>8}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    tr_loss  = run_epoch(loader_train, train=True)
    val_loss = run_epoch(loader_val,   train=False)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    cur_lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t_start

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6}  {tr_loss:>12.6f}  {val_loss:>12.6f}  {cur_lr:>10.2e}  {elapsed:>7.1f}s")

# Ripristina il miglior modello
model.load_state_dict(best_state)
torch.save(best_state, OUT_DIR / "gnn_ae_weights.pt")
print(f"\nMiglior val loss: {best_val_loss:.6f}  — pesi salvati in outputs/GNN/gnn_ae_weights.pt")


 Epoch    Train Loss      Val Loss          LR      Time
------------------------------------------------------------
     1      0.230123      0.043974    9.99e-04    221.4s


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_x = np.arange(1, len(train_losses)+1)

axes[0].plot(epochs_x, train_losses, label="Train", color="#1f77b4")
axes[0].plot(epochs_x, val_losses,   label="Val",   color="#ff7f0e")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Loss — GNN Autoencoder"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(epochs_x, train_losses, label="Train", color="#1f77b4")
axes[1].semilogy(epochs_x, val_losses,   label="Val",   color="#ff7f0e")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE Loss (log)")
axes[1].set_title("Loss — scala log"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "gnn_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: outputs/GNN/gnn_loss.png")


## 5. Anomaly Score

Per ogni jet l'**anomaly score** è la media del MSE di ricostruzione per traccia:
$$\text{score}(\text{jet}) = \frac{1}{|\mathcal{T}_{\text{valid}}|} \sum_{i \in \mathcal{T}_{\text{valid}}} \|x_i - \hat{x}_i\|^2$$


In [ ]:
@torch.no_grad()
def compute_scores(loader) -> np.ndarray:
    """
    Ritorna un array (N_jets,) con l'anomaly score per ogni jet.
    score = media MSE per traccia (media sul batch gestita da global_mean_pool).
    """
    model.eval()
    all_scores = []

    for batch in loader:
        batch = batch.to(DEVICE)
        x_rec, z, bvec = model(batch)

        # Per-track squared error (media su N_FEAT)
        per_track_mse = ((x_rec - batch.x) ** 2).mean(dim=1)  # (N_nodes,)

        # Mean per jet
        per_jet_score = global_mean_pool(
            per_track_mse.unsqueeze(1), batch.batch
        ).squeeze(1)  # (B,)

        all_scores.append(per_jet_score.cpu().numpy())

    return np.concatenate(all_scores)


print("Calcolo anomaly scores...")
scores_qcd = compute_scores(loader_qtest)
scores_ej  = compute_scores(loader_etest)

print(f"QCD test — score: mean={scores_qcd.mean():.5f}  std={scores_qcd.std():.5f}  median={np.median(scores_qcd):.5f}")
print(f"EJ  test — score: mean={scores_ej.mean():.5f}  std={scores_ej.std():.5f}  median={np.median(scores_ej):.5f}")
print(f"Separazione (ratio mediane): {np.median(scores_ej)/np.median(scores_qcd):.2f}x")


In [ ]:
COLOR_QCD = "#1f77b4"  # blu
COLOR_EJ  = "#d62728"  # rosso

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scala lineare
combined = np.concatenate([scores_qcd, scores_ej])
lo, hi   = np.percentile(combined, [0.5, 99.5])
bins     = np.linspace(lo, hi, 80)

for ax, yscale in zip(axes, ["linear", "log"]):
    ax.hist(scores_qcd, bins=bins, density=True, alpha=0.65, color=COLOR_QCD,
            label=f"QCD (n={len(scores_qcd):,})")
    ax.hist(scores_ej,  bins=bins, density=True, alpha=0.65, color=COLOR_EJ,
            label=f"EJ  (n={len(scores_ej):,})")
    ax.axvline(np.median(scores_qcd), color=COLOR_QCD, ls="--", lw=1.5, label=f"mediana QCD={np.median(scores_qcd):.4f}")
    ax.axvline(np.median(scores_ej),  color=COLOR_EJ,  ls="--", lw=1.5, label=f"mediana EJ={np.median(scores_ej):.4f}")
    ax.set_xlabel("Anomaly Score (MSE per traccia)")
    ax.set_ylabel("Densità" + (" (log)" if yscale=="log" else ""))
    ax.set_yscale(yscale)
    ax.set_title(f"Distribuzione Anomaly Score — GNN AE ({yscale})")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "gnn_scores_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: outputs/GNN/gnn_scores_dist.png")


## 6. ROC Curve e AUC

In [ ]:
# Costruisci il dataset di test: EJ=anomalo (positive), QCD=normale (negative)
y_true  = np.concatenate([np.zeros(len(scores_qcd)), np.ones(len(scores_ej))])
y_score = np.concatenate([scores_qcd, scores_ej])

fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = sk_auc(fpr, tpr)

# Efficienza EJ al mistag rate del 1% e 10%
def eff_at_fpr(target_fpr):
    idx = np.searchsorted(fpr, target_fpr)
    idx = min(idx, len(tpr)-1)
    return tpr[idx]

eff_1pct  = eff_at_fpr(0.01)
eff_10pct = eff_at_fpr(0.10)

print(f"AUC (GNN AE) = {roc_auc:.4f}")
print(f"Efficienza EJ @ 1%  mistag QCD: {eff_1pct:.3f}")
print(f"Efficienza EJ @ 10% mistag QCD: {eff_10pct:.3f}")

# ── Plot ROC ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC standard
axes[0].plot(fpr, tpr, color="#2ca02c", lw=2,
             label=f"GNN AE  (AUC={roc_auc:.4f})")
axes[0].plot([0,1],[0,1], "k--", lw=1, label="Random")
axes[0].scatter([0.01, 0.10], [eff_1pct, eff_10pct],
                zorder=5, color="red", s=60)
axes[0].set_xlabel("False Positive Rate (QCD mistag)")
axes[0].set_ylabel("True Positive Rate (EJ efficiency)")
axes[0].set_title("ROC Curve — GNN Autoencoder")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

# Curva ROC log-log (stile fisica delle particelle: 1/FPR vs TPR)
with np.errstate(divide="ignore", invalid="ignore"):
    rej = np.where(fpr > 0, 1.0 / fpr, np.nan)
axes[1].plot(tpr, rej, color="#2ca02c", lw=2, label=f"GNN AE  (AUC={roc_auc:.4f})")
axes[1].set_xscale("linear"); axes[1].set_yscale("log")
axes[1].set_xlabel("EJ Efficiency (TPR)")
axes[1].set_ylabel("QCD Rejection (1/FPR)")
axes[1].set_title("Efficiency vs QCD Rejection — GNN AE")
axes[1].legend(); axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig(OUT_DIR / "gnn_roc.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: outputs/GNN/gnn_roc.png")


## 7. Salvataggio risultati

In [ ]:
# Salva scores per confronto con altri modelli (AE flat, VAE)
np.savez(
    OUT_DIR / "gnn_test_scores.npz",
    scores_qcd = scores_qcd,
    scores_ej  = scores_ej,
    fpr        = fpr,
    tpr        = tpr,
    roc_auc    = roc_auc,
)

# Salva config modello
model_cfg = {
    "architecture"  : "GNN EdgeConv Autoencoder",
    "n_feat"        : N_FEAT,
    "hidden_dims"   : HIDDEN_DIMS,
    "latent_dim"    : LATENT_DIM,
    "knn_k"         : KNN_K,
    "epochs"        : EPOCHS,
    "lr"            : LR,
    "weight_decay"  : WEIGHT_DECAY,
    "batch_size"    : BATCH_SIZE,
    "n_params"      : n_params,
    "roc_auc"       : float(roc_auc),
    "eff_1pct_fpr"  : float(eff_1pct),
    "eff_10pct_fpr" : float(eff_10pct),
    "best_val_loss" : float(best_val_loss),
}
with open(OUT_DIR / "gnn_model_config.json", "w") as f:
    json.dump(model_cfg, f, indent=2)

print("File salvati in outputs/GNN/:")
for p in sorted(OUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<35} {size_kb:>8.1f} KB")


## 8. Confronto con AE flat e VAE (opzionale)

Se il notebook `06_autoencoder.ipynb` è già stato eseguito, confrontiamo le ROC.


In [ ]:
ae_scores_path = BASE_DIR / "outputs" / "ae_out" / "test_scores.npz"

if ae_scores_path.exists():
    ae = np.load(ae_scores_path)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # GNN AE
    ax.plot(fpr, tpr, color="#2ca02c", lw=2, label=f"GNN AE  (AUC={roc_auc:.4f})")
    
    # AE flat
    if "scores_qcd_ae" in ae and "scores_ej_ae" in ae:
        y_true_ae  = np.concatenate([np.zeros(len(ae["scores_qcd_ae"])), np.ones(len(ae["scores_ej_ae"]))])
        y_score_ae = np.concatenate([ae["scores_qcd_ae"], ae["scores_ej_ae"]])
        fpr_ae, tpr_ae, _ = roc_curve(y_true_ae, y_score_ae)
        auc_ae = sk_auc(fpr_ae, tpr_ae)
        ax.plot(fpr_ae, tpr_ae, color="#1f77b4", lw=2, ls="--", label=f"AE flat (AUC={auc_ae:.4f})")
    
    # VAE flat
    if "scores_qcd_vae" in ae and "scores_ej_vae" in ae:
        y_true_v  = np.concatenate([np.zeros(len(ae["scores_qcd_vae"])), np.ones(len(ae["scores_ej_vae"]))])
        y_score_v = np.concatenate([ae["scores_qcd_vae"], ae["scores_ej_vae"]])
        fpr_v, tpr_v, _ = roc_curve(y_true_v, y_score_v)
        auc_v = sk_auc(fpr_v, tpr_v)
        ax.plot(fpr_v, tpr_v, color="#ff7f0e", lw=2, ls="-.", label=f"VAE     (AUC={auc_v:.4f})")
    
    ax.plot([0,1],[0,1], "k--", lw=1, label="Random")
    ax.set_xlabel("FPR (QCD mistag)"); ax.set_ylabel("TPR (EJ efficiency)")
    ax.set_title("Confronto ROC: GNN AE vs AE flat vs VAE")
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    
    plt.tight_layout()
    plt.savefig(OUT_DIR / "roc_confronto_modelli.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Salvato: outputs/GNN/roc_confronto_modelli.png")
else:
    print(f"File {ae_scores_path} non trovato — esegui prima il notebook 06_autoencoder.")
    print("Il confronto richiede i scores dell\'AE flat salvati in outputs/ae_out/test_scores.npz")


## Riepilogo

| Modello | AUC | Eff EJ @1% FPR |
|---------|-----|----------------|
| **GNN AE** (questo notebook) | vedi `gnn_model_config.json` | vedi `gnn_model_config.json` |

### Scelte architetturali
- **EdgeConv**: cattura relazioni locali tra tracce vicine in (Δη, Δφ), naturale per jet come point cloud
- **Global Mean Pool**: aggrega l'informazione di tutte le tracce nel vettore latente del jet
- **Skip connection** (concat encoder + latent al decoder): migliora la qualità di ricostruzione
- **k-NN k=8**: bilancio tra connettività locale e costo computazionale

### Interpretazione fisica
- Tracce EJ hanno `radiusOfFirstHit` più grande e `numberOfInnermostPixelLayerHits` = 0  
- Il GNN addestrato su QCD non riesce a ricostruire bene questi pattern → score alto  
- La struttura a grafo conserva la topologia spaziale del jet, non disponibile nei modelli flat

### File di output (`outputs/GNN/`)
- `gnn_ae_weights.pt` — pesi del modello
- `gnn_loss.png` — curve di loss
- `gnn_scores_dist.png` — distribuzioni anomaly score
- `gnn_roc.png` — ROC curve
- `gnn_test_scores.npz` — scores grezzi per analisi ulteriori
- `gnn_model_config.json` — iperparametri e metriche
